In [2]:

import os, json, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.base import clone
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

warnings.filterwarnings("ignore")

REPO = Path.cwd()
for _ in range(8):
    if (REPO / "homework11").exists() or (REPO / ".git").exists():
        break
    REPO = REPO.parent

STAGE_DIR = REPO / "homework11"
NB_DIR    = STAGE_DIR / "notebooks"
REP_DIR   = STAGE_DIR / "reports"
ART_DIR   = STAGE_DIR / "artifacts"
PLOTS     = ART_DIR / "plots"
METRICS   = ART_DIR / "metrics"
for d in [NB_DIR, REP_DIR, PLOTS, METRICS]:
    d.mkdir(parents=True, exist_ok=True)

print(f"[setup] Outputs -> {STAGE_DIR}")


def discover_dataset():
    candidates = []
    for base in [REPO / "data", REPO / "project" / "data", STAGE_DIR]:
        if base.exists():
            candidates += list(base.rglob("*.csv")) + list(base.rglob("*.parquet"))
    candidates = sorted(candidates, key=lambda p: p.stat().st_size)
    for p in candidates:
        try:
            df = pd.read_parquet(p) if p.suffix.lower()==".parquet" else pd.read_csv(p)
            if df.shape[0] >= 50 and df.shape[1] >= 2:
                return p, df
        except Exception:
            pass
    return None, None

path, df = discover_dataset()
if path is None:
    print("[data] No dataset found → using synthetic fallback.")
    rng = np.random.default_rng(42)
    n = 320
    risk = rng.normal(50, 10, n)
    rev  = 40000 + 120*risk + rng.normal(0, 3000, n)
    df = pd.DataFrame({"revenue": rev, "risk_index_var95": risk})
    path = ART_DIR / "synthetic_homework11.csv"
    df.to_csv(path, index=False)
else:
    print(f"[data] Using dataset: {path}")

# Choose target; default to 'revenue' if present
POSSIBLE_TARGETS = ["revenue", "target", "y", "ret_next", "y_next"]
TARGET = next((c for c in POSSIBLE_TARGETS if c in df.columns), df.columns[-1])

# Numeric X except target; y = target
X = df.select_dtypes(include=[np.number]).drop(columns=[TARGET], errors="ignore").copy()
y = df[TARGET].copy()
assert len(X) == len(y) and len(X) >= 50, "Not enough rows or X/y mismatch."

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"[data] TARGET={TARGET} | X={X.shape} | train={len(X_train)} test={len(X_test)}")


def _as_np(a): return np.asarray(a)
def _rmse(y_true, y_pred):
    yt, yp = _as_np(y_true), _as_np(y_pred)
    return float(np.sqrt(mean_squared_error(yt, yp)))

def regression_metrics(y_true, y_pred, p=None):
    yt, yp = _as_np(y_true), _as_np(y_pred)
    rmse = _rmse(yt, yp)
    mae  = float(mean_absolute_error(yt, yp))
    r2   = float(r2_score(yt, yp))
    n    = len(yt)
    try:
        p = int(p if p is not None else (X_test.shape[1] if hasattr(X_test, "shape") else 1))
    except Exception:
        p = 1
    adj_r2 = 1 - (1 - r2) * (n - 1) / max(n - p - 1, 1)
    return {"n": int(n), "rmse": rmse, "mae": mae, "r2": r2, "adj_r2": float(adj_r2)}

def bootstrap_ci_rmse(y_true, y_pred, B=1000, alpha=0.05, seed=42):
    yt, yp = _as_np(y_true), _as_np(y_pred)
    n = len(yt)
    idx = np.arange(n)
    rng = np.random.default_rng(seed)
    vals = []
    for _ in range(B):
        s = rng.choice(idx, size=n, replace=True)  # positional resample
        vals.append(_rmse(yt[s], yp[s]))
    lo, hi = np.quantile(vals, [alpha/2, 1-alpha/2])
    return float(np.mean(vals)), float(lo), float(hi)

def _to_py(obj):
    """Recursively convert numpy/pandas (incl. Interval/Timestamp) to plain Python for JSON."""
    import numpy as _np, pandas as _pd
    if isinstance(obj, (str, int, float, bool)) or obj is None: return obj
    if isinstance(obj, (_np.integer, _np.floating, _np.bool_)): return obj.item()
    if isinstance(obj, (_pd.Timestamp, _pd.Timedelta, _pd.Period, _pd.Interval)): return str(obj)
    if isinstance(obj, _np.ndarray): return obj.tolist()
    if isinstance(obj, dict): return {str(k): _to_py(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple, set)): return [_to_py(v) for v in obj]
    if isinstance(obj, pd.Series):
        return _to_py(obj.apply(lambda x: str(x) if isinstance(x, pd.Interval) else x).to_dict())
    if isinstance(obj, pd.DataFrame):
        dfc = obj.copy()
        for col in dfc.columns:
            if pd.api.types.is_interval_dtype(dfc[col]): dfc[col] = dfc[col].astype(str)
            elif dfc[col].dtype == "object":
                dfc[col] = dfc[col].apply(lambda x: str(x) if isinstance(x, pd.Interval) else x)
        if isinstance(dfc.index, pd.IntervalIndex): dfc.index = dfc.index.astype(str)
        return _to_py(dfc.to_dict(orient="records"))
    return str(obj)

def safe_json_dump(payload, path):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(_to_py(payload), f, indent=2, ensure_ascii=False)


pipe = Pipeline([
    ("imp", SimpleImputer(strategy="median")),
    ("sc",  StandardScaler()),
    ("lr",  LinearRegression())
])
pipe.fit(X_train, y_train)
yhat = pipe.predict(X_test)

overall = regression_metrics(y_test, yhat, p=X_test.shape[1])
rmse_mean, ci_lo, ci_hi = bootstrap_ci_rmse(y_test, yhat, B=1200)
overall.update({"rmse_boot_mean": rmse_mean, "rmse_ci_low": ci_lo, "rmse_ci_high": ci_hi})
print("[eval] overall:", {k:(round(v,4) if isinstance(v,(int,float)) else v) for k,v in overall.items()})


def run_variant(strategy: str):
    mdl = Pipeline([
        ("imp", SimpleImputer(strategy=strategy)),
        ("sc",  StandardScaler()),
        ("lr",  LinearRegression())
    ])
    mdl.fit(X_train, y_train)
    pred = mdl.predict(X_test)
    m = regression_metrics(y_test, pred, p=X_test.shape[1])
    m["imputer"] = strategy
    return m

scenarios = pd.DataFrame([run_variant("mean"), run_variant("median")])
scenarios.to_csv(METRICS / "scenario_compare.csv", index=False)

df_eval = pd.DataFrame(X_test).reset_index(drop=True)
df_eval["y_true"] = _as_np(y_test).reshape(-1)
df_eval["y_pred"] = _as_np(yhat).reshape(-1)
df_eval["resid"]  = df_eval["y_true"] - df_eval["y_pred"]

diagnostics = {}

if "risk_index_var95" in df_eval.columns:
    df_eval["risk_q"] = pd.qcut(df_eval["risk_index_var95"], q=5, duplicates="drop").astype(str)
    seg = df_eval.groupby("risk_q").apply(lambda d: regression_metrics(d["y_true"], d["y_pred"])).apply(pd.Series)
    ax = seg["rmse"].plot(kind="bar", title="RMSE by Risk Quintile")
    ax.figure.tight_layout(); ax.figure.savefig(PLOTS / "rmse_by_risk_quintile.png", dpi=160); plt.close(ax.figure)
    diagnostics["by_risk_quintile"] = seg.reset_index().to_dict(orient="records")

if "revenue" in df_eval.columns:  # only if revenue ends up in features
    df_eval["rev_size"] = pd.qcut(df_eval["revenue"].abs(), q=5, duplicates="drop").astype(str)
    seg2 = df_eval.groupby("rev_size").apply(lambda d: regression_metrics(d["y_true"], d["y_pred"])).apply(pd.Series)
    ax2 = seg2["rmse"].plot(kind="bar", title="RMSE by Revenue Size")
    ax2.figure.tight_layout(); ax2.figure.savefig(PLOTS / "rmse_by_rev_size.png", dpi=160); plt.close(ax2.figure)
    diagnostics["by_revenue_size"] = seg2.reset_index().to_dict(orient="records")


payload = {
    "overall": overall,
    "scenarios": scenarios,     # DataFrame OK; safe_json_dump handles it
    "segments": diagnostics
}
safe_json_dump(payload, METRICS / "eval_summary.json")

# Residuals vs Predicted
plt.figure()
plt.scatter(df_eval["y_pred"], df_eval["resid"], s=18, alpha=0.7)
plt.axhline(0, ls="--")
plt.xlabel("Predicted"); plt.ylabel("Residual")
plt.title("Residuals vs Predicted")
plt.tight_layout(); plt.savefig(PLOTS / "residuals_vs_pred.png", dpi=160); plt.close()

# Residual distribution
plt.figure()
pd.Series(df_eval["resid"]).hist(bins=20)
plt.xlabel("Residual"); plt.ylabel("Count")
plt.title("Residual Distribution")
plt.tight_layout(); plt.savefig(PLOTS / "residual_hist.png", dpi=160); plt.close()

# Predicted vs Actual
mn = float(min(df_eval["y_true"].min(), df_eval["y_pred"].min()))
mx = float(max(df_eval["y_true"].max(), df_eval["y_pred"].max()))
plt.figure()
plt.scatter(df_eval["y_true"], df_eval["y_pred"], s=18, alpha=0.7)
plt.plot([mn, mx], [mn, mx], ls="--")
plt.xlabel("Actual"); plt.ylabel("Predicted")
plt.title("Predicted vs Actual")
plt.tight_layout(); plt.savefig(PLOTS / "pred_vs_actual.png", dpi=160); plt.close()

print("[save] metrics:", METRICS / "eval_summary.json")
print("[save] scenarios:", METRICS / "scenario_compare.csv")
print("[save] plots:", [p.name for p in PLOTS.glob("*.png")])


def walk_forward_backtest(model, X_all: pd.DataFrame, y_all: np.ndarray, initial: int, step: int):
    rows = []; i = initial
    while i < len(y_all):
        tr = np.arange(0, i)
        te = np.arange(i, min(i+step, len(y_all)))
        m = clone(model); m.fit(X_all.iloc[tr], y_all[tr])
        pred = m.predict(X_all.iloc[te])
        rows.append({"fold_end": int(te[-1]), "rmse": _rmse(y_all[te], pred), "n_test": int(len(te))})
        i += step
    return pd.DataFrame(rows)

try:
    X_all = pd.concat([pd.DataFrame(X_train), pd.DataFrame(X_test)], axis=0).reset_index(drop=True)
    y_all = np.concatenate([_as_np(y_train), _as_np(y_test)])
    bt = walk_forward_backtest(pipe, X_all, y_all, initial=max(20, len(y_all)//3), step=max(5, len(y_all)//10))
    bt.to_csv(METRICS / "backtest_rmse.csv", index=False)
    plt.figure(); plt.plot(bt["fold_end"], bt["rmse"], marker="o")
    plt.xlabel("Fold End (time order proxy)"); plt.ylabel("RMSE"); plt.title("Walk-Forward RMSE Over Time")
    plt.tight_layout(); plt.savefig(PLOTS / "backtest_rmse_over_time.png", dpi=160); plt.close()
    print("[save] backtest:", METRICS / "backtest_rmse.csv")
except Exception as e:
    print("[backtest] skipped:", e)


rep = REP_DIR / "homework11_evaluation.md"
rep.write_text(f"""# Homework 11 — Evaluation & Risk Communication

**Summary**
- RMSE = {overall['rmse']:.2f}, MAE = {overall['mae']:.2f}, R² = {overall['r2']:.3f}
- Bootstrap RMSE 95% CI: [{overall['rmse_ci_low']:.2f}, {overall['rmse_ci_high']:.2f}] (B=1200)
- Scenarios compared (mean vs median imputation). See: `artifacts/metrics/scenario_compare.csv`

**Diagnostics**
- See `artifacts/plots/`: `pred_vs_actual.png`, `residuals_vs_pred.png`, `residual_hist.png`
- If available: `rmse_by_risk_quintile.png`, `rmse_by_rev_size.png`
- (Optional) `backtest_rmse_over_time.png`

**Assumptions & Risks**
- Assumes data distribution similar to training; sensitive to extreme volatility/missing-rate > 10%.
- Misuse risk: treat predictions as ranges, not exact points.

**Go/No-Go (example)**
- RMSE ≤ target; 95% empirical PI coverage ≥ 90%;
- No subgroup RMSE > 1.5× overall; no sustained degradation in backtest.

**Artifacts**
- Metrics JSON: `artifacts/metrics/eval_summary.json`
- Scenario table: `artifacts/metrics/scenario_compare.csv`
- Plots: `artifacts/plots/*.png`
""", encoding="utf-8")
print("[save] report:", rep)


[setup] Outputs -> C:\Users\User\bootcamp_Khushi_Khanna\homework\homework11
[data] Using dataset: C:\Users\User\bootcamp_Khushi_Khanna\homework\homework11\artifacts\synthetic_homework11.csv
[data] TARGET=revenue | X=(320, 1) | train=256 test=64
[eval] overall: {'n': 64, 'rmse': 3115.8588, 'mae': 2449.5413, 'r2': 0.0006, 'adj_r2': -0.0155, 'rmse_boot_mean': 3101.6982, 'rmse_ci_low': 2550.7138, 'rmse_ci_high': 3653.3881}
[save] metrics: C:\Users\User\bootcamp_Khushi_Khanna\homework\homework11\artifacts\metrics\eval_summary.json
[save] scenarios: C:\Users\User\bootcamp_Khushi_Khanna\homework\homework11\artifacts\metrics\scenario_compare.csv
[save] plots: ['backtest_rmse_over_time.png', 'pred_vs_actual.png', 'residuals_vs_pred.png', 'residual_hist.png', 'rmse_by_risk_quintile.png']
[save] backtest: C:\Users\User\bootcamp_Khushi_Khanna\homework\homework11\artifacts\metrics\backtest_rmse.csv
[save] report: C:\Users\User\bootcamp_Khushi_Khanna\homework\homework11\reports\homework11_evaluation